# 论文里的图怎样画才好：坐标轴、颜色、不确定性与导出

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/Paper-Figures/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

本篇用五组“坏图”与“好图”的对比，讲最重要的几条原则。Matplotlib 的基本用法见[Matplotlib 入门](https://wxrkz123.github.io/blog/Python-Matplotlib/)。所有的图都由下面的代码生成：

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import wooldridge

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "PingFang SC", "Noto Sans CJK SC", "WenQuanYi Zen Hei"]   # 中文字体，按顺序找到第一个可用的
plt.rcParams["axes.unicode_minus"] = False          # 正常显示负号
plt.rcParams["axes.spines.top"] = False             # 去掉上边框和右边框
plt.rcParams["axes.spines.right"] = False
print("ok")

## 一、柱状图的纵轴必须从 0 开始

柱状图用柱子的**长度**表示数值，读者会不自觉地比较长度的比例。纵轴不从 0 开始，长度的比例就被扭曲了：

In [ ]:
groups, values = ["对照组", "处理组"], [102, 105]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].bar(groups, values, color=["#9e9e9e", "#2171b5"])
axes[0].set_ylim(100, 106)
axes[0].set_title("坏：纵轴从 100 开始，差异看起来有 2.5 倍")
axes[1].bar(groups, values, color=["#9e9e9e", "#2171b5"])
axes[1].set_ylim(0, 120)
for x, v in enumerate(values):
    axes[1].text(x, v + 2, f"{v}", ha="center")
axes[1].set_title("好：纵轴从 0 开始，并标出数值")
for ax in axes:
    ax.set_ylabel("平均工资指数")
fig.tight_layout()
fig.savefig("bars.png", dpi=150)

## 二、双纵轴可以制造任何“相关”

两条序列画在同一张图上、左右各用一个纵轴，看起来很直观，其实非常容易误导：两个纵轴的范围都可以任意选择，通过调整它们，任意两条有趋势的序列都可以被画得“几乎重合”：

In [ ]:
rng = np.random.default_rng(1)
years = np.arange(2000, 2025)
a = 50 + np.cumsum(rng.normal(1.0, 1.5, len(years)))        # 两条毫无关系、各自有上升趋势的序列
b = 3 + np.cumsum(rng.normal(0.2, 0.15, len(years)))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax2 = axes[0].twinx()
axes[0].plot(years, a, color="#2171b5")
ax2.plot(years, b, color="#e6550d")
ax2.spines["right"].set_visible(True)
axes[0].set_ylabel("序列 A", color="#2171b5")
ax2.set_ylabel("序列 B", color="#e6550d")
axes[0].set_title("坏：双纵轴，两条无关的序列“高度同步”")

axes[1].plot(years, a / a[0] * 100, color="#2171b5", label="序列 A")
axes[1].plot(years, b / b[0] * 100, color="#e6550d", label="序列 B")
axes[1].text(years[-1] + 0.3, a[-1] / a[0] * 100, "序列 A", color="#2171b5", va="center")
axes[1].text(years[-1] + 0.3, b[-1] / b[0] * 100, "序列 B", color="#e6550d", va="center")
axes[1].set_ylabel("指数（2000 年 = 100）")
axes[1].set_title("好：换算成同一基期的指数")
fig.tight_layout()
fig.savefig("dual_axis.png", dpi=150)
print(f"两条序列的相关系数 {np.corrcoef(a, b)[0, 1]:.2f}；一阶差分的相关系数 {np.corrcoef(np.diff(a), np.diff(b))[0, 1]:.2f}")

## 三、大样本的散点图：用分箱散点图

几万个点的散点图会糊成一团，看不出任何关系。经济学论文中常用**分箱散点图**（binned scatter plot，binscatter）：按横轴变量的分位数把样本分成 20 组左右，画出每组的横轴均值和纵轴均值：

In [ ]:
rng = np.random.default_rng(2)
n = 20000
age = rng.uniform(22, 60, n)
log_wage = 2.0 + 0.06 * (age - 22) - 0.0012 * (age - 22) ** 2 + rng.normal(0, 0.55, n)
df = pd.DataFrame({"age": age, "log_wage": log_wage})

bins = df.groupby(pd.qcut(df["age"], 20), observed=True).agg(age=("age", "mean"), log_wage=("log_wage", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
axes[0].scatter(df["age"], df["log_wage"], s=4, color="#2171b5")
axes[0].set_title(f"坏：{n:,} 个点糊成一团")
axes[1].scatter(df["age"], df["log_wage"], s=3, color="#c6dbef", alpha=0.3, rasterized=True)
axes[1].scatter(bins["age"], bins["log_wage"], s=30, color="#08519c", zorder=3)
axes[1].set_title("好：分箱散点图（20 个等分组的均值）")
axes[1].set_ylim(1.5, 4.5)
for ax in axes:
    ax.set_xlabel("年龄")
axes[0].set_ylabel("对数工资")
fig.tight_layout()
fig.savefig("binscatter.png", dpi=150)

## 四、系数图：画出不确定性，并说明是哪种区间

回归结果除了用表格，也常用**系数图**展示：每个系数一个点，加上置信区间。用伍德里奇教材的工资数据：

In [ ]:
wage1 = wooldridge.data("wage1")
fit = smf.ols("lwage ~ educ + exper + tenure + female + married + nonwhite", data=wage1).fit(cov_type="HC1")
names = {"educ": "受教育年限", "exper": "工作经验", "tenure": "现职年限", "female": "女性", "married": "已婚", "nonwhite": "非白人"}
coef = fit.params.drop("Intercept").rename(names)
ci = fit.conf_int().drop("Intercept").rename(index=names)

fig, ax = plt.subplots(figsize=(6, 3.6))
y = np.arange(len(coef))[::-1]
ax.errorbar(coef, y, xerr=[coef - ci[0], ci[1] - coef], fmt="o", color="#2171b5", capsize=3)
ax.axvline(0, color="#999", lw=1, ls="--")
ax.set_yticks(y, coef.index)
ax.set_xlabel("对对数工资的影响（点：系数；线：95% 置信区间，稳健标准误）")
ax.set_title(f"工资方程的系数（n = {int(fit.nobs)}）")
fig.tight_layout()
fig.savefig("coefplot.png", dpi=150)
print(coef.round(3).to_dict())

## 五、颜色与图例

In [ ]:
rng = np.random.default_rng(3)
t = np.arange(2010, 2025)
series = {name: 100 * np.cumprod(1 + rng.normal(mu, 0.03, len(t)))
          for name, mu in [("东部", 0.06), ("中部", 0.05), ("西部", 0.045), ("东北", 0.02)]}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for (name, s), c in zip(series.items(), ["red", "green", "orange", "yellow"]):
    axes[0].plot(t, s, color=c, label=name, lw=1.5)
axes[0].legend(loc="upper left", ncol=2)
axes[0].set_title("坏：红绿搭配、黄色看不清、要来回对照图例")

palette = ["#08306b", "#2171b5", "#6baed6", "#e6550d"]       # 三个同色系 + 一个强调色
for (name, s), c in zip(series.items(), palette):
    axes[1].plot(t, s, color=c, lw=2 if name == "东北" else 1.5)
    axes[1].text(t[-1] + 0.3, s[-1], name, color=c, va="center", fontsize=10)
axes[1].set_xlim(t[0], t[-1] + 2)
axes[1].set_title("好：直接标注、只突出要讲的那一条")
for ax in axes:
    ax.set_xlabel("年份")
axes[0].set_ylabel("人均收入指数（2010 年 = 100）")
fig.tight_layout()
fig.savefig("colors.png", dpi=150)

## 六、导出：格式、字号与字体

**格式**：论文中的图优先保存为**矢量格式**（PDF、SVG、EPS）：任意放大都清晰，文字可以被搜索。PNG 等位图格式至少用 300 dpi。Word 中插入图片时，新版本的 Word 支持 SVG。

（示意代码，不在笔记本中运行）

```python
fig.savefig("figure1.pdf", bbox_inches="tight")              # 矢量格式，裁掉多余的白边
fig.savefig("figure1.png", dpi=300, bbox_inches="tight")     # 位图，300 dpi
```